# 多头注意力机制代码
## 注意力计算
![注意力计算](./images/attention.png)
1. 计算Q和K的相似度 得到score
$$
scores=QK^T
$$
2. 计算Attention，对scores做归一化，得到概率分布
$$
attention=softmax(\frac{QK^T}{\sqrt{d_k}})
$$
加系数$d_k$的原因：避免 $QK^T$ 的数值计算结果过大，导致 $\text{softmax}$ 向着梯度很小的区域偏移
3. score和v相乘 得到O
$$
O=softmax(\frac{QK^T}{\sqrt{d_k}})V
$$
## 多头注意力
定义头数num_heads 要求能被model_dim整除

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from einops import rearrange

# RMSNorm
LayerNorm：
$$\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

相比 LayerNorm，RMSNorm 省去了计算 $\mu$ 和 $\beta$ 的开销，只用均方根做缩放：
$$\text{RMSNorm}(x) = \dfrac{x}{\text{RMS}(x)} \cdot \gamma,\quad \text{RMS}(x)=\sqrt{\dfrac{1}{d}\sum x_i^2 + \epsilon}$$

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # TODO: 定义可学习的缩放参数 γ，形状为 (dim,)，初始值全为 1
        # 提示：使用 nn.Parameter(torch.ones(dim))
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (B, T, dim)
        x_normed = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True).add(self.eps))

        return x_normed * self.weight

In [3]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention (Vaswani et al., "Attention Is All You Need", 2017)

    核心思路：
      将 Q/K/V 投影到 h 个低维子空间，各自计算 Scaled Dot-Product Attention，
      再把结果拼接后做一次线性投影输出。

    维度约定（全文统一）：
      B  = batch size
      T  = sequence length (token 数)
      d  = model dimension (d_model)
      h  = number of heads
      d_k = d // h  (每个 head 的维度)
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 定义Wq,Wk,Wv,Wo
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    # 定义点积
    def scaled_dot_product_attention(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        mask: torch.Tensor = None,
    ):
        # q (batch_size, num_heads, seq_len,d_k)
        # V^T (batch_size,num_heads,d_k,seq_len)
        # 计算scores （batch_size,num_heads,
        scores = torch.matmul(q, k.transpose(-2, -1))
        # 缩放
        scores /= math.sqrt(self.d_k)

        # 设置掩码
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        # 计算注意力
        weights = F.softmax(scores, dim=-1)

        # 计算输出
        output = torch.matmul(self.dropout(weights), v)

        return output, weights

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        freqs_cis: torch.Tensor = None,
        mask: torch.Tensor = None,
    ):
        B, T_q, _ = query.shape
        _, T_k, _ = key.shape

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # 拆分多头
        Q = rearrange(
            Q,
            "batch seq (heads head_dim) -> batch heads seq head_dim",
            heads=self.num_heads,
        )
        K = rearrange(
            K,
            "batch seq (heads head_dim) -> batch heads seq head_dim",
            heads=self.num_heads,
        )
        V = rearrange(
            V,
            "batch seq (heads head_dim) -> batch heads seq head_dim",
            heads=self.num_heads,
        )

        if freqs_cis is not None:
            Q, K = apply_rotary_emb(Q, K, freqs_cis)

        # 计算注意力
        # [RoPE] 在点积前转动 Q 和 K
        if freqs_cis is not None:
            # 仅截取与当前 T_q, T_k 对应的 freqs_cis (支持变长)
            Q, K = apply_rotary_emb(
                Q, K, freqs_cis[:T_q]
            )  # 假设 T_q == T_k (Self-Attention) 或依需要截断

        attn_output, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)

        # Step 4: 拼接多头
        # 先 transpose 回 (B, T_q, h, d_k)，再 contiguous().reshape → (B, T_q, d_model)
        # NOTE: 必须先调 contiguous()，因为 transpose 后内存不连续，reshape 会报错
        attn_output = rearrange(
            attn_output, "batch heads seq head_dim -> batch seq (heads head_dim)"
        )

        output = self.W_o(attn_output)

        return output, attn_weights

In [4]:
class MultiHeadAttention_Reference(nn.Module):
    """Reference implementation — DO NOT peek during the interview!"""

    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, q, k, v, mask=None):
        scores = torch.matmul(q, k.transpose(-2, -1))  # (B,h,T_q,T_k)
        scores = scores / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        output = torch.matmul(weights, v)  # (B,h,T_q,d_k)
        return output, weights

    def forward(self, query, key, value, mask=None):
        B, T_q, _ = query.shape
        _, T_k, _ = key.shape
        Q = self.W_q(query)  # (B,T_q,d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        Q = Q.reshape(B, T_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K.reshape(B, T_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V.reshape(B, T_k, self.num_heads, self.d_k).transpose(1, 2)
        attn_output, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        attn_output = (
            attn_output.transpose(1, 2).contiguous().reshape(B, T_q, self.d_model)
        )
        output = self.W_o(attn_output)
        return output, attn_weights

In [5]:
torch.manual_seed(42)
B, T, d_model, h = 2, 10, 64, 8

mha = MultiHeadAttention(d_model, h, dropout=0.0)
x = torch.randn(B, T, d_model)

# Self-attention (Q=K=V=x)
out, weights = mha(x, x, x)
print(f"Output shape : {out.shape}")  # expect (2, 10, 64)
print(f"Weights shape: {weights.shape}")  # expect (2, 8, 10, 10)

# Cross-entropy sanity: output should match reference
ref = MultiHeadAttention_Reference(d_model, h)
ref.load_state_dict(mha.state_dict())
ref_out, _ = ref(x, x, x)
print(f"Max diff vs reference: {(out - ref_out).abs().max().item():.6f}")  # expect ~0

Output shape : torch.Size([2, 10, 64])
Weights shape: torch.Size([2, 8, 10, 10])
Max diff vs reference: 0.000000


# 位置编码

## 正弦位置编码
$$
\begin{align} PE(pos,2i)=sin(pos/10000^{2i/d_{model}}) \\ PE(pos,2i+1)=cos(pos/10000^{2i/d_{model}}) \end{align}
$$

In [6]:
# 正弦位置编码
def pos_sinusoid_embedding(d_model: int, seq_len: int):
    embeddings = torch.zeros((seq_len, d_model))
    for i in range(d_model):
        f = torch.sin if i % 2 == 0 else torch.cos
        # 给embeddings的第i列复制
        embeddings[:, i] = f(
            torch.arange(0, seq_len) / np.power(1e4, 2 * (i // 2) / d_model)
        )
    return embeddings.float()


## 旋转位置编码 RoPE（Rotary Position Embedding）
RoPE 不把位置信息加到 token embedding 上，而是直接旋转 Q 和 K：
$$q_m^\top k_n = \text{Re}\left[\sum_j \mathbf{W}_j \cdot e^{i(m-n)\theta_j}\right]$$
核心操作：将 $q$ / $k$ 中每对相邻维度视为复数，乘以 $e^{im\theta_j}$，使得点积天然感知相对位置差 $(m-n)$。

In [7]:
def precompute_freqs_cis(
    dim: int, seq_len: int, theta: float = 10000.0
) -> torch.Tensor:
    """
    预计算旋转位置编码所需的复数频率矩阵。
    返回形状: (seq_len, dim // 2)，dtype=torch.complex64

    每个位置 m，每个维度对 j 对应：e^{i * m * θ_j}
    其中 θ_j = 1 / (theta^(2j / dim))
    """
    # 第一步：计算各维度对的频率 θ_j，形状 (dim//2,)
    # 提示：torch.arange(0, dim, 2).float() / dim → 再用 theta ** (-...) 得到 freqs
    freqs = theta ** -(torch.arange(0, dim, 2).float() / dim)

    # 第二步：生成位置索引 t = [0, 1, ..., seq_len-1]，形状 (seq_len,)
    t = torch.arange(0, seq_len)

    # 第三步：外积 t ⊗ freqs，得到每个位置×维度的角度矩阵，形状 (seq_len, dim//2)
    # 提示：torch.outer(t, freqs)
    freqs_mat = torch.outer(t, freqs)

    # 第四步：转为复数形式 e^{i*freqs_mat} = cos + i*sin，形状 (seq_len, dim//2)
    # 提示：torch.polar(torch.ones_like(freqs_mat), freqs_mat)
    freqs_cis = torch.polar(torch.ones_like(freqs_mat), freqs_mat)

    return freqs_cis


def apply_rotary_emb(
    q: torch.Tensor,  # (B, num_heads, T, head_dim)
    k: torch.Tensor,  # (B, num_heads, T, head_dim)
    freqs_cis: torch.Tensor,  # (T, head_dim // 2)
):
    """
    将预计算好的旋转因子作用到 Q 和 K 上。
    """
    # 第一步：把 q/k 最后一维相邻两两配对，reshape 成复数视图
    # 提示：torch.view_as_complex(x.float().reshape(..., -1, 2))
    # q shape 变化: (B, heads, T, head_dim) → (B, heads, T, head_dim//2) 复数
    q_complex = torch.view_as_complex(
        rearrange(q.float(), "b h t (d r) -> b h t d r ", r=2)
    )
    k_complex = torch.view_as_complex(
        rearrange(k.float(), "b h t (d r) -> b h t d r ", r=2)
    )
    # 第二步：对齐 freqs_cis 维度以便广播
    # freqs_cis 形状 (T, head_dim//2) → 需要变成 (1, 1, T, head_dim//2)
    freqs_cis = rearrange(freqs_cis, "t d-> 1 1 t d")

    # 第三步：复数相乘（等价于旋转），再转回实数并展平最后两维
    # 提示：torch.view_as_real(x * freqs_cis).flatten(-2)
    q_out = rearrange(
        torch.view_as_real(q_complex * freqs_cis), "b h t d r -> b h t (d r)"
    )
    k_out = rearrange(
        torch.view_as_real(k_complex * freqs_cis), "b h t d r -> b h t (d r)"
    )

    # 转回原始 dtype 并返回
    return q_out.type_as(q), k_out.type_as(k)

# 因果掩码 Causal Mask

In [8]:
def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """
    生成因果掩码（上三角为 True，表示需要遮挡的未来位置）

    返回形状: (1, 1, seq_len, seq_len)
    广播适配: (B, num_heads, seq_len, seq_len)
    """
    casual_mask = torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1
    )
    return casual_mask.unsqueeze(0).unsqueeze(0)

# 位置掩码 Padding Mask

In [9]:
def make_padding_mask(token_ids: torch.Tensor, pad_id: int = 0) -> torch.Tensor:
    """
    生成padding掩码，True 表示该位置是 PAD，需要遮挡

    参数：
        token_ids: (B, T)
        pad_id: 填充的token id

    返回形状: (B, 1, 1, T)
    广播适配: (B, num_heads, T_query, T)
    """
    # 找出 PAD 位置
    padding_mask = token_ids == pad_id
    return padding_mask.unsqueeze(1).unsqueeze(2)

# 逐位置前馈网络（Position-wise Feed-Forward Networks）

## 基于 ReLU 的 FFN
$$\text{FFN}(x) = \max(0,\ xW_1 + b_1)W_2 + b_2$$

In [10]:
# ── 原 ReLU FFN（已替换为 SwiGLU，保留备查）────────────────────────
# class PositionwiseFFN(nn.Module):
#     """
#     Position-wise Feed-Forward Network
#
#     FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
#
#     维度变化：
#       d_model → d_ff → d_model
#       通常 d_ff = 4 × d_model（原论文：512 → 2048 → 512）
#     """
#
#     def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
#         super().__init__()
#         self.W1 = nn.Linear(d_model, d_ff)
#         self.W2 = nn.Linear(d_ff, d_model)
#         self.dropout = nn.Dropout(dropout)
#         self.relu = nn.ReLU()
#
#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         x = self.W1(x)
#         x = self.relu(x)
#         x = self.dropout(x)
#         x = self.W2(x)
#         return x


## 基于 SwiGLU 的 FFN 
SwiGLU 不像普通激活函数那样只对一个线性结果激活，而是**分别做了两个线性投影，对其中一个激活后，再和另一个相乘（门控 Gating）**：
$$ \text{FFN}_{\text{SwiGLU}}(x) = \big( \text{SiLU}(xW_1) \otimes xV \big) W_2 $$
通常为了保持总参数量，隐藏层维度 $d_{ff}$ 会调整为 $\frac{8}{3} d_{model}$，但这里保留外部传入的维度即可。

In [11]:
class PositionwiseFFN(nn.Module):
    """
    基于 SwiGLU 的 Position-wise Feed-Forward Network

    FFN(x) = (SiLU(xW1) ⊗ xV)W2
    注意：大模型通常不使用 bias
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: 定义三个线性层 (不使用 bias)
        # 1. 门控线性层 Gate (对应公式的 xW1)
        self.W1 = nn.Linear(d_model, d_ff, bias=False)
        # 2. 上投影 Up (对应公式的 xV)
        self.V = nn.Linear(d_model, d_ff, bias=False)
        # 3. 下投影 Down (对应公式最后乘的 W2)
        self.W2 = nn.Linear(d_ff, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 第一步：计算门控激活后的结果 SiLU(xW1)
        # 提示：使用 F.silu(...)
        gate = F.silu(self.W1(x))

        # 第二步：将门控结果与 xV 逐元素相乘
        up_proj = self.V(x)
        activated = gate * up_proj

        # 可选：如果需要 Dropout，可以在此处应用
        activated = self.dropout(activated)

        # 第三步：计算最终的下投影
        out = self.W2(activated)

        return out

# 编码器（Encoder）

In [12]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        # self.norm1 = nn.LayerNorm(d_model)
        self.norm1 = RMSNorm(d_model)
        # self.norm2 = nn.LayerNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor = None, freqs_cis: torch.Tensor = None
    ):
        # x:(B,T,d_model)
        # 原版无 RoPE:
        # attn_out, _ = self.self_attn(x, x, x, mask)
        attn_out, _ = self.self_attn(x, x, x, freqs_cis, mask)
        x = self.norm1(x + self.dropout(attn_out))  # 残差＋归一化
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x

In [13]:
class Encoder(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        num_layers: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )  # 原论文为6层
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor = None, freqs_cis: torch.Tensor = None
    ):
        x = self.dropout(x)
        for layer in self.layers:
            # x = layer(x, mask)  # 原版
            x = layer(x, mask, freqs_cis)
        return x

# 解码器（Decoder）

In [14]:
class DecoderLayer(nn.Module):
    """
    单个编码器层，包含三个子层：
      1. Masked Self-Attention
      2. Cross-Attention (Q 来自 Encoder, K/V 来自 Decoder)
      3. Position-wise FFN
    每个子层后接 残差连接 + LayerNorm
    """

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        # self.norm1 = nn.LayerNorm(d_model)
        self.norm1 = RMSNorm(d_model)
        # self.norm2 = nn.LayerNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        # self.norm3 = nn.LayerNorm(d_model)
        self.norm3 = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        tgt: torch.Tensor,  # 解码器输入 (B,T_tgt, d_model)
        enc_out: torch.Tensor,  # 编码器输出 (B, T_src, d_model)
        tgt_mask: torch.Tensor = None,  # casual mask (1, 1, T_tgt, T_tgt)
        src_mask: torch.Tensor = None,  # padding mask (B, 1, 1, T_src)
        freqs_cis: torch.Tensor = None,
    ):
        # self_attn_out, _ = self.self_attn(tgt, tgt, tgt, tgt_mask)  # 原版
        self_attn_out, _ = self.self_attn(tgt, tgt, tgt, freqs_cis, tgt_mask)
        tgt = self.norm1(tgt + self.dropout(self_attn_out))
        # Cross-Attention 不使用 RoPE，因为 src 和 tgt 是不同的序列，相对位置没有直接对比意义
        cross_attn_out, _ = self.cross_attn(tgt, enc_out, enc_out, src_mask)
        tgt = self.norm2(tgt + self.dropout(cross_attn_out))
        ffn_out = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout(ffn_out))

        return tgt

In [15]:
class Decoder(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        num_layers: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

    def forward(
        self,
        tgt: torch.Tensor,
        enc_out: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        src_mask: torch.Tensor = None,
        freqs_cis: torch.Tensor = None,
    ) -> torch.Tensor:
        for layer in self.layers:
            # tgt = layer(tgt, enc_out, tgt_mask, src_mask)  # 原版
            tgt = layer(tgt, enc_out, tgt_mask, src_mask, freqs_cis)

        return tgt

# Transformer
![transformer](./images/transformer.png)


In [16]:
class Embedding(nn.Module):
    def __init__(
        self, vocab_size: int, d_model: int, max_seq_len: int, dropout: float = 0.1
    ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        # （原正弦PE已移除）RoPE 不在 Embedding 层添加，而是在 Attention 中作用于 Q/K
        # pe = pos_sinusoid_embedding(d_model, max_seq_len)
        # self.register_buffer("pe", pe)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.token_embedding(x)
        # x = x + self.pe[: x.size(1), :]
        x = self.dropout(x)
        return x

In [17]:
class Transformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        num_layers: int,
        max_seq_len: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.embedding = Embedding(vocab_size, d_model, max_seq_len, dropout)

        # [RoPE] 预先计算好最大长度的旋转频率矩阵，作为 Buffer (不参与梯度更新)
        # 这里 head_dim = d_model // num_heads
        head_dim = d_model // num_heads
        freqs_cis = precompute_freqs_cis(head_dim, max_seq_len)
        self.register_buffer("freqs_cis", freqs_cis)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)

        self.out_proj = nn.Linear(d_model, vocab_size)

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,
        src_mask: torch.Tensor = None,
        tgt_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        # enc_out = self.encoder(self.embedding(src), src_mask)  # 原版
        # dec_out = self.decoder(self.embedding(tgt), enc_out, tgt_mask, src_mask)  # 原版

        # [RoPE] 截取当前批次中最长的序列对应的频率分片
        freqs_src = self.freqs_cis[: src.size(1)]
        freqs_tgt = self.freqs_cis[: tgt.size(1)]

        enc_out = self.encoder(self.embedding(src), src_mask, freqs_src)
        dec_out = self.decoder(
            self.embedding(tgt), enc_out, tgt_mask, src_mask, freqs_tgt
        )
        logits = self.out_proj(dec_out)

        return logits

In [18]:
# [验证] 完整的 Transformer (包含 RoPE) 前向传播测试
if __name__ == "__main__":
    B, src_len, tgt_len = 2, 50, 40
    vocab_size, d_model, n_heads = 1000, 64, 8

    model = Transformer(
        vocab_size=vocab_size,
        d_model=d_model,
        num_heads=n_heads,
        d_ff=256,
        num_layers=2,
        max_seq_len=100,
    )

    src = torch.randint(0, vocab_size, (B, src_len))
    tgt = torch.randint(0, vocab_size, (B, tgt_len))

    # Casual Mask
    tgt_mask = make_causal_mask(tgt_len, src.device)

    out = model(src, tgt, tgt_mask=tgt_mask)
    print(
        "✅ RoPE Transformer Forward Pass Successful! Output shape:", out.shape
    )  # Exepect (2, 40, 1000)

✅ RoPE Transformer Forward Pass Successful! Output shape: torch.Size([2, 40, 1000])
